In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [8]:
train_data = torch.load("../data/cmnist/train_biased.pt")
test_biased = torch.load("../data/cmnist/test_biased.pt")
test_unbiased = torch.load("../data/cmnist/test_unbiased.pt")

In [9]:
# understanding the data first
print(type(train_data))
print(len(train_data))
print(train_data[0])

<class 'tuple'>
2
tensor([[[[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]],

         [[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]],

         [[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]]],


        [[[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0.,

In [10]:
# Split into images and labels
train_images, train_labels = train_data
test_biased_images, test_biased_labels = test_biased
test_unbiased_images, test_unbiased_labels = test_unbiased

# Create datasets
train_dataset = TensorDataset(train_images, train_labels)
test_biased_dataset = TensorDataset(test_biased_images, test_biased_labels)
test_unbiased_dataset = TensorDataset(test_unbiased_images, test_unbiased_labels)

# Create loaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_biased_loader = DataLoader(test_biased_dataset, batch_size=64, shuffle=False)
test_unbiased_loader = DataLoader(test_unbiased_dataset, batch_size=64, shuffle=False)

In [11]:
class CustomCNN(nn.Module):
    def __init__(self):
        super(CustomCNN, self).__init__()
        
        self.conv1 = nn.Conv2d(3, 8, kernel_size=3, padding=1)  # changed to 3
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        
        self.pool = nn.MaxPool2d(2,2)
        
        self.fc1 = nn.Linear(32*3*3, 64)
        self.fc2 = nn.Linear(64, 10)
        
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.pool(self.relu(self.conv3(x)))
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [12]:
model = CustomCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [13]:
num_epochs = 10  
train_losses, train_acc = [], []
val_losses, val_acc = [], []

for epoch in range(num_epochs):
    model.train()  
    running_loss = 0.0
    correct = 0
    total = 0
    
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()       
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    train_losses.append(running_loss / total)
    train_acc.append(correct / total)
    
    
    model.eval()  
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():  
        for images, labels in test_biased_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
    
    val_losses.append(val_loss / val_total)
    val_acc.append(val_correct / val_total)
    
    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {train_losses[-1]:.4f}, Train Acc: {train_acc[-1]:.4f} "
          f"Val Loss: {val_losses[-1]:.4f}, Val Acc: {val_acc[-1]:.4f}")

Epoch [1/10] Train Loss: 0.2958, Train Acc: 0.9195 Val Loss: 0.0836, Val Acc: 0.9762
Epoch [2/10] Train Loss: 0.0703, Train Acc: 0.9793 Val Loss: 0.0525, Val Acc: 0.9838
Epoch [3/10] Train Loss: 0.0478, Train Acc: 0.9865 Val Loss: 0.0355, Val Acc: 0.9889
Epoch [4/10] Train Loss: 0.0372, Train Acc: 0.9893 Val Loss: 0.0258, Val Acc: 0.9918
Epoch [5/10] Train Loss: 0.0306, Train Acc: 0.9910 Val Loss: 0.0273, Val Acc: 0.9919
Epoch [6/10] Train Loss: 0.0269, Train Acc: 0.9918 Val Loss: 0.0270, Val Acc: 0.9919
Epoch [7/10] Train Loss: 0.0220, Train Acc: 0.9928 Val Loss: 0.0284, Val Acc: 0.9913
Epoch [8/10] Train Loss: 0.0200, Train Acc: 0.9940 Val Loss: 0.0213, Val Acc: 0.9928
Epoch [9/10] Train Loss: 0.0174, Train Acc: 0.9945 Val Loss: 0.0247, Val Acc: 0.9922
Epoch [10/10] Train Loss: 0.0143, Train Acc: 0.9957 Val Loss: 0.0183, Val Acc: 0.9946


In [15]:
def evaluate(loader):
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    return correct / total

In [16]:
biased_acc = evaluate(test_biased_loader)
unbiased_acc = evaluate(test_unbiased_loader)

print("Biased Accuracy:", biased_acc)
print("Unbiased Accuracy:", unbiased_acc)

Biased Accuracy: 0.9946
Unbiased Accuracy: 0.9295


#### 3. Analytical Question 1.3: Report the accuracies for both test sets. You should observe a significant drop in performance on the unbiased test set. Explain this phenomenon mathematically and conceptually. Why did the optimizer prefer the color feature over the shape feature during training?

here the model achieved a biased test accuracy of 0.9946 and an unbiased test accuracy of 0.9295. 


Normally, I  expected the unbiased accuracy to drop significantly because the model tends to exploit the color as an easy shortcut during training.

 Mathematically, the optimizer minimise the loss by finding features that have the strongest correlation with the label but in this case, color is easier to pick up than shape, so the network initially relies on it: it learns a higher P(label l color) rather than P(label l shape).
 
  Conceptually, this happens because CNN can detect correlations that are consistent in the training set even if they are false. As a result, when the unbiased test set removes the color correlation, the model performs worse, showing its reliance on color over shape.

#### 4. Analytical Question 1.4: Propose training strategies that could force the network to ignore the color and focus on the shape of the digits (You are not required to provide code for this).

One simple strategy is to convert all training images to grayscale before feeding them to the model. This completely removes color information so the model has no choice but to learn the shape of the digits, its the most straightforward way to eliminate the shortcut.


Another strategy is that I use color augmentation or color jitter during training, where the colors of images are randomly changed or shuffled every epoch. 